# PII Detection & Classification Report

This notebook runs the project model and a baseline (Presidio) on `ai_data.csv`, then generates tables and plots for your report.

In [ ]:
# Install dependencies (run once)
!python -m pip install presidio-analyzer matplotlib seaborn pandas scikit-learn tabulate
!python -m spacy download en_core_web_sm


In [ ]:
import os, sys, ast
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


In [ ]:
# Paths
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))  # repo root from ml_report_project
DATA_PATH = os.path.join(os.getcwd(), 'data', 'ai_data.csv')
OUT_DIR = os.path.join(os.getcwd(), 'outputs')
PLOTS_DIR = os.path.join(OUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

print('DATA_PATH:', DATA_PATH)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)
print(df.columns)
df.head(2)


In [ ]:
# Utilities
def normalize_text(value: str) -> str:
    if value is None:
        return ''
    text = str(value).strip().strip('"').strip("'")
    return ' '.join(text.lower().split())

def load_gold_labels(raw_value):
    if raw_value is None or (isinstance(raw_value, float) and pd.isna(raw_value)):
        return []
    if isinstance(raw_value, dict):
        data = raw_value
    else:
        data = ast.literal_eval(str(raw_value))

    gold = []
    for label, items in data.items():
        if items is None:
            continue
        if isinstance(items, (list, tuple)):
            clean_items = [normalize_text(x) for x in items if normalize_text(x)]
            for item in clean_items:
                gold.append((item, label))
            if len(clean_items) > 1:
                joined = normalize_text(' '.join(clean_items))
                if joined:
                    gold.append((joined, label))
        else:
            item = normalize_text(items)
            if item:
                gold.append((item, label))
    return gold

def build_gold_maps(gold_list):
    gold_set = set()
    gold_labels_by_text = defaultdict(set)
    for text, label in gold_list:
        gold_set.add(text)
        gold_labels_by_text[text].add(label)
    return gold_set, gold_labels_by_text

def map_project_label(label: str):
    mapping = {
        'EMAIL': 'EMAIL',
        'PHONE': 'PHONE_NUM',
        'URL': 'URL_PERSONAL',
        'PERSON_NAME': 'NAME_STUDENT',
        'ADDRESS': 'STREET_ADDRESS',
        'IP_ADDRESS': 'ID_NUM',
        'ACCOUNT_ID': 'ID_NUM',
        'SSN': 'ID_NUM',
        'IBAN': 'ID_NUM',
    }
    return mapping.get(label)

def map_presidio_label(label: str):
    mapping = {
        'EMAIL_ADDRESS': 'EMAIL',
        'PHONE_NUMBER': 'PHONE_NUM',
        'URL': 'URL_PERSONAL',
        'PERSON': 'NAME_STUDENT',
        'USERNAME': 'USERNAME',
        'LOCATION': 'STREET_ADDRESS',
        'CREDIT_CARD': 'ID_NUM',
        'US_SSN': 'ID_NUM',
        'IP_ADDRESS': 'ID_NUM',
    }
    return mapping.get(label)

def compute_metrics(rows):
    tp = sum(r['tp'] for r in rows)
    fp = sum(r['fp'] for r in rows)
    fn = sum(r['fn'] for r in rows)
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1, tp, fp, fn


In [ ]:
# Load project model
sys.path.insert(0, ROOT)
from apps.api.anonymizer import PIIAnonymizer
project_model = PIIAnonymizer()


In [ ]:
# Load baseline (Presidio)
from presidio_analyzer import AnalyzerEngine
presidio = AnalyzerEngine()


In [ ]:
# Run evaluation
per_row_detection = {"project": [], "presidio": []}
per_row_classification = {"project": [], "presidio": []}
per_label_counts = {
    "project": defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0}),
    "presidio": defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0}),
}
pred_type_counts = {"project": Counter(), "presidio": Counter()}
gold_type_counts = Counter()
sample_rows = []
text_lengths = []
gold_counts_per_text = []
pred_counts_per_text = {"project": [], "presidio": []}

for _, row in df.iterrows():
    text = str(row['0'])
    text_lengths.append(len(text))
    gold = load_gold_labels(row['1'])
    gold_set, gold_labels_by_text = build_gold_maps(gold)
    gold_counts_per_text.append(len(gold_set))
    for _, lbl in gold:
        gold_type_counts[lbl] += 1

    project_raw = project_model.detect_pii(text)
    project_preds = [(normalize_text(t), map_project_label(lbl), lbl) for t, lbl, _, _ in project_raw]
    project_preds = [(t, mapped, raw) for t, mapped, raw in project_preds if t]

    presidio_raw = presidio.analyze(text=text, language="en")
    presidio_preds = [
        (normalize_text(text[r.start:r.end]), map_presidio_label(r.entity_type), r.entity_type)
        for r in presidio_raw
    ]
    presidio_preds = [(t, mapped, raw) for t, mapped, raw in presidio_preds if t]

    for t, mapped, raw in project_preds:
        pred_type_counts['project'][mapped or raw] += 1
    for t, mapped, raw in presidio_preds:
        pred_type_counts['presidio'][mapped or raw] += 1

    project_pred_texts = {t for t, _, _ in project_preds}
    pred_counts_per_text['project'].append(len(project_pred_texts))
    presidio_pred_texts = {t for t, _, _ in presidio_preds}
    pred_counts_per_text['presidio'].append(len(presidio_pred_texts))

    for name, pred_texts in [("project", project_pred_texts), ("presidio", presidio_pred_texts)]:
        tp = len(pred_texts & gold_set)
        fp = len(pred_texts - gold_set)
        fn = len(gold_set - pred_texts)
        per_row_detection[name].append({"tp": tp, "fp": fp, "fn": fn})

    for name, preds in [("project", project_preds), ("presidio", presidio_preds)]:
        tp = fp = fn = 0
        matched_gold = set()
        for t, mapped_label, _ in preds:
            if not mapped_label:
                continue
            gold_labels = gold_labels_by_text.get(t, set())
            if mapped_label in gold_labels:
                tp += 1
                matched_gold.add((t, mapped_label))
                per_label_counts[name][mapped_label]["tp"] += 1
            else:
                fp += 1
                per_label_counts[name][mapped_label]["fp"] += 1

        for t, labels in gold_labels_by_text.items():
            for label in labels:
                if (t, label) not in matched_gold:
                    fn += 1
                    per_label_counts[name][label]["fn"] += 1
        per_row_classification[name].append({"tp": tp, "fp": fp, "fn": fn})

    if len(sample_rows) < 5:
        sample_rows.append({
            'text': text[:200],
            'gold_labels': '; '.join(sorted({lbl for _, lbl in gold})),
            'project_pred_types': '; '.join(sorted({mapped or raw for _, mapped, raw in project_preds})),
            'presidio_pred_types': '; '.join(sorted({mapped or raw for _, mapped, raw in presidio_preds})),
        })


In [ ]:
# Build summary tables
summary_rows = []
for name in ['project', 'presidio']:
    p, r, f1, tp, fp, fn = compute_metrics(per_row_detection[name])
    summary_rows.append({
        'model': name,
        'metric': 'detection_text_match',
        'precision': p,
        'recall': r,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
    })
    p, r, f1, tp, fp, fn = compute_metrics(per_row_classification[name])
    summary_rows.append({
        'model': name,
        'metric': 'classification_text_and_type',
        'precision': p,
        'recall': r,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
per_label_rows = []
for name in ['project', 'presidio']:
    for label, counts in per_label_counts[name].items():
        tp = counts['tp']; fp = counts['fp']; fn = counts['fn']
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        per_label_rows.append({
            'model': name,
            'label': label,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'fp': fp,
            'fn': fn,
        })

per_label_df = pd.DataFrame(per_label_rows)
per_label_df.head(10)


In [ ]:
# Type count table
type_counts_df = pd.DataFrame({
    'label': sorted(set(list(gold_type_counts.keys()) + list(pred_type_counts['project'].keys()) + list(pred_type_counts['presidio'].keys()))),
})
type_counts_df['gold_count'] = type_counts_df['label'].map(gold_type_counts).fillna(0).astype(int)
type_counts_df['project_pred_count'] = type_counts_df['label'].map(pred_type_counts['project']).fillna(0).astype(int)
type_counts_df['presidio_pred_count'] = type_counts_df['label'].map(pred_type_counts['presidio']).fillna(0).astype(int)
type_counts_df.head(10)


In [ ]:
# Save tables
summary_df.to_csv(os.path.join(OUT_DIR, 'summary_metrics.csv'), index=False)
per_label_df.to_csv(os.path.join(OUT_DIR, 'per_label_metrics.csv'), index=False)
type_counts_df.to_csv(os.path.join(OUT_DIR, 'type_counts.csv'), index=False)
pd.DataFrame(sample_rows).to_csv(os.path.join(OUT_DIR, 'sample_predictions.csv'), index=False)
summary_df


In [ ]:
# Plot: overall F1
plt.figure(figsize=(7, 4))
plot_df = summary_df.copy()
plot_df['model'] = plot_df['model'].str.title()
sns.barplot(data=plot_df, x='metric', y='f1', hue='model')
plt.xticks(rotation=20, ha='right')
plt.title('Overall F1 Comparison')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'overall_f1.png'), dpi=200)
plt.show()


In [ ]:
# Plot: per-label F1
if not per_label_df.empty:
    plt.figure(figsize=(9, 4.5))
    plot_df = per_label_df.copy()
    plot_df['model'] = plot_df['model'].str.title()
    sns.barplot(data=plot_df, x='label', y='f1', hue='model')
    plt.xticks(rotation=30, ha='right')
    plt.title('Per-Label F1 (Text + Type Match)')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'per_label_f1.png'), dpi=200)
    plt.show()


In [ ]:
# Macro-average table
macro_rows = []
for name in ['project', 'presidio']:
    model_df = per_label_df[per_label_df['model'] == name]
    macro_rows.append({
        'model': name,
        'metric': 'macro_avg',
        'precision': model_df['precision'].mean() if not model_df.empty else 0.0,
        'recall': model_df['recall'].mean() if not model_df.empty else 0.0,
        'f1': model_df['f1'].mean() if not model_df.empty else 0.0,
    })
macro_df = pd.DataFrame(macro_rows)
macro_df


In [ ]:
# Dataset stats
if 'gold_counts_per_text' not in globals():
    gold_counts_per_text = []
if 'pred_counts_per_text' not in globals():
    pred_counts_per_text = {'project': [], 'presidio': []}

dataset_stats = pd.DataFrame([
    {'stat': 'num_rows', 'value': len(df)},
    {'stat': 'avg_text_length', 'value': df['0'].astype(str).str.len().mean()},
    {'stat': 'avg_gold_entities_per_text', 'value': sum(gold_counts_per_text) / max(len(gold_counts_per_text), 1)},
    {'stat': 'avg_project_entities_per_text', 'value': sum(pred_counts_per_text['project']) / max(len(pred_counts_per_text['project']), 1)},
    {'stat': 'avg_presidio_entities_per_text', 'value': sum(pred_counts_per_text['presidio']) / max(len(pred_counts_per_text['presidio']), 1)},
])
dataset_stats


In [ ]:
# Plot: label support (gold vs predictions)
plot_df = type_counts_df.copy()
plot_df = plot_df.melt(id_vars=['label'], value_vars=['gold_count','project_pred_count','presidio_pred_count'],
                   var_name='source', value_name='count')
plt.figure(figsize=(9, 4.5))
sns.barplot(data=plot_df, x='label', y='count', hue='source')
plt.xticks(rotation=30, ha='right')
plt.title('PII Label Support (Gold vs Predictions)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'label_support.png'), dpi=200)
plt.show()


In [ ]:
# Plot: text length distribution
plt.figure(figsize=(7, 4))
sns.histplot(df['0'].astype(str).str.len(), bins=20, kde=True)
plt.title('Text Length Distribution')
plt.xlabel('Characters')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'text_length_distribution.png'), dpi=200)
plt.show()


In [ ]:
# Plot: entities per text
plt.figure(figsize=(7, 4))
sns.histplot(gold_counts_per_text, bins=15, color='black', label='gold', kde=False)
sns.histplot(pred_counts_per_text['project'], bins=15, color='blue', label='project', kde=False, alpha=0.6)
sns.histplot(pred_counts_per_text['presidio'], bins=15, color='orange', label='presidio', kde=False, alpha=0.6)
plt.legend()
plt.title('Entities per Text')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'entities_per_text.png'), dpi=200)
plt.show()


In [ ]:
# Save extended outputs
macro_df.to_csv(os.path.join(OUT_DIR, 'macro_metrics.csv'), index=False)
dataset_stats.to_csv(os.path.join(OUT_DIR, 'dataset_stats.csv'), index=False)
pd.DataFrame(error_samples['project']).to_csv(os.path.join(OUT_DIR, 'project_error_samples.csv'), index=False)
pd.DataFrame(error_samples['presidio']).to_csv(os.path.join(OUT_DIR, 'presidio_error_samples.csv'), index=False)
macro_df


In [ ]:
# View error samples (qualitative analysis)
pd.DataFrame(error_samples['project']).head(10)


In [ ]:
pd.DataFrame(error_samples['presidio']).head(10)
